# A — pre-run validation (NO full run)
GPU T4 x2, Internet ON, secret HF_TOKEN attached. ~15 min. Produces `prerun_validation.zip` with:
model_check.json (exact Meta revision, tokenizer + chat template), truncation_report.json (choice of max_new_tokens),
timing + extraction sanity on the 60-row sample. **The full run (notebook B) starts only after the checklist is approved.**

In [ ]:
import os, shutil, glob, json, subprocess
hits = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/FROZEN_v2.json', recursive=True)]
assert hits, 'attach the rag-noise-probe dataset (with artifacts/data/FROZEN_v2.json)'
SRC = os.path.dirname(os.path.dirname(hits[0])); REPO = '/kaggle/working/rag-noise-probe'
shutil.copytree(SRC, REPO, dirs_exist_ok=True); os.chdir(REPO)
!pip install -q rank_bm25 pyarrow pytest 2>&1 | tail -1
!python -m pytest tests -q 2>&1 | tail -2
import transformers, torch; print('transformers', transformers.__version__, 'torch', torch.__version__)
!nvidia-smi --query-gpu=name,memory.total --format=csv
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
_tok, _label = None, None
for _l in ['HF_TOKEN', 'hf_token', 'HF_token', 'Hf_token', 'HUGGINGFACE_TOKEN', 'HUGGING_FACE_HUB_TOKEN', 'huggingface_token', 'HF']:
    try:
        _tok = UserSecretsClient().get_secret(_l); _label = _l; break
    except Exception as e:
        print('secret label', repr(_l), '-> not available', type(e).__name__)
assert _tok, 'No Hugging Face secret reachable from this run (attach it under Add-ons -> Secrets)'
print('using secret label:', repr(_label))   # the value is never printed
login(token=_tok); del _tok
SECRET_ONLY = os.environ.get('SECRET_ONLY') == '1'
MODEL = 'meta-llama/Llama-2-7b-chat-hf'          # revision is pinned in artifacts/data/model_pin.json
BENCH = 'artifacts/data/benchmark_v2.jsonl'
RUN = 'artifacts/runs/llama2-7b'; os.makedirs(RUN, exist_ok=True)

## 1. Model identity: pinned revision, file fingerprints (incl. weights), chat template, prompt rendering

In [ ]:
!python -m src.gen.model_check --bench $BENCH --out $RUN/prerun/model_check.json --hash_weights

## 2. Truncation sample: 60 stratified uids (12/condition), both templates, generous limit 160

In [ ]:
U = 'artifacts/data/trunc_sample_uids.txt'
!python -m src.gen.gen --model $MODEL --bench $BENCH --uids $U --template std     --max_new_tokens 160 --out $RUN/prerun/gen_std_160.jsonl --checkpoint_every 20
!python -m src.gen.gen --model $MODEL --bench $BENCH --uids $U --template abstain --max_new_tokens 160 --out $RUN/prerun/gen_abstain_160.jsonl --checkpoint_every 20
!python -m src.gen.truncation_check report --gen $RUN/prerun/gen_std_160.jsonl $RUN/prerun/gen_abstain_160.jsonl --bench $BENCH --out $RUN/prerun/truncation_report.json

## 3. Extraction sanity + timing on the std sample (tf_match must be ~1.0; heads from authors' topk_heads.json)

In [ ]:
!python -m src.extract.redeep_extract --model $MODEL --gen $RUN/prerun/gen_std_160.jsonl --bench $BENCH --outdir $RUN/prerun --checkpoint_every 20
import pandas as pd, numpy as np
f = pd.read_parquet(f'{RUN}/prerun/redeep_feats.parquet'); m = json.load(open(f'{RUN}/prerun/redeep_meta.json'))
g = [json.loads(l) for l in open(f'{RUN}/prerun/gen_std_160.jsonl')]
print('tf_match', f.tf_match.mean(), '| head_source', m['head_source'], '| revision', m.get('revision'))
print('gen s/row', np.mean([r['gen_seconds'] for r in g]).round(2))
for r in g[:12]: print(r['uid'].split('__')[1], r['n_answer_tokens'], r['stopped_by_eos'], '|', repr(r['answer'][:110]))

In [ ]:
import zipfile
with zipfile.ZipFile('/kaggle/working/prerun_validation.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for root in ['artifacts/runs/llama2-7b/prerun']:
        for dp, dn, fn in os.walk(root):
            if '.parts' in dp: continue
            for n in fn:
                p = os.path.join(dp, n); z.write(p, p.replace(os.sep, '/'))
print(os.path.getsize('/kaggle/working/prerun_validation.zip') / 1e6, 'MB')